In [1]:
import os
import pandas as pd

# List files in the current working directory
print("Current directory:", os.getcwd())
print("\nAvailable files:")

for file in os.listdir():
    print(file)



df = pd.read_csv("../data/Sample - Superstore.csv", encoding="cp1252")

print("Dataset loaded successfully")
print("Shape:", df.shape)

display(df.head())

Current directory: d:\consulting\Retail Profitability Turnaround\notebook

Available files:
01_load_data.ipynb
02_analyis.ipynb
Dataset loaded successfully
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


Our first analytical question
Before we investigate discounts, products or regions, we need to answer:
Is the retailer's profitability problem concentrated in specific parts of the business, or is it widespread?
This determines where we spend our analytical effort. We will first decompose total profit, then segment it.

## Quantify the profit pool

In [2]:
# Separate profitable and loss-making records
profitable = df[df["Profit"] > 0]
loss_making = df[df["Profit"] < 0]
break_even = df[df["Profit"] == 0]

total_profit_generated = profitable["Profit"].sum()
total_losses = loss_making["Profit"].sum()  # Negative number

profit_reconciliation = pd.DataFrame({
    "Metric": [
        "Total sales",
        "Profit from profitable records",
        "Losses from loss-making records",
        "Net profit",
        "Number of profitable records",
        "Number of loss-making records",
        "Number of break-even records"
    ],
    "Value": [
        df["Sales"].sum(),
        total_profit_generated,
        total_losses,
        df["Profit"].sum(),
        len(profitable),
        len(loss_making),
        len(break_even)
    ]
})

display(profit_reconciliation)

,Metric,Value
0,Total sales,2.297201e+06
1,Profit from profitable records,4.425283e+05
2,Losses from loss-making records,-1.561313e+05
3,Net profit,2.863970e+05
4,Number of profitable records,8.058000e+03
5,Number of loss-making records,1.871000e+03
6,Number of break-even records,6.500000e+01


In [3]:
def profitability_summary(data, group_column):
    result = (
        data.groupby(group_column)
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Records=("Profit", "size"),
            Loss_Making_Records=(
                "Profit", lambda x: (x < 0).sum()
            )
        )
        .reset_index()
    )

    result["Profit_Margin_%"] = (
        result["Profit"] / result["Sales"] * 100
    )

    result["Sales_Share_%"] = (
        result["Sales"] / data["Sales"].sum() * 100
    )

    result["Loss_Record_Share_%"] = (
        result["Loss_Making_Records"] / result["Records"] * 100
    )

    return result.sort_values("Profit", ascending=True)


print("REGIONAL PROFITABILITY")
display(profitability_summary(df, "Region").round(2))

print("CATEGORY PROFITABILITY")
display(profitability_summary(df, "Category").round(2))

print("CUSTOMER SEGMENT PROFITABILITY")
display(profitability_summary(df, "Segment").round(2))

REGIONAL PROFITABILITY


,Region,Sales,Profit,Records,Loss_Making_Records,Profit_Margin_%,Sales_Share_%,Loss_Record_Share_%
0,Central,501239.89,39706.36,2323,741,7.92,21.82,31.90
2,South,391721.90,46749.43,1620,259,11.93,17.05,15.99
1,East,678781.24,91522.78,2848,553,13.48,29.55,19.42
3,West,725457.82,108418.45,3203,318,14.94,31.58,9.93


CATEGORY PROFITABILITY


,Category,Sales,Profit,Records,Loss_Making_Records,Profit_Margin_%,Sales_Share_%,Loss_Record_Share_%
0,Furniture,741999.80,18451.27,2121,714,2.49,32.3,33.66
1,Office Supplies,719047.03,122490.80,6026,886,17.04,31.3,14.70
2,Technology,836154.03,145454.95,1847,271,17.40,36.4,14.67


CUSTOMER SEGMENT PROFITABILITY


,Segment,Sales,Profit,Records,Loss_Making_Records,Profit_Margin_%,Sales_Share_%,Loss_Record_Share_%
2,Home Office,429653.15,60298.68,1783,312,14.03,18.70,17.50
1,Corporate,706146.37,91979.13,3020,556,13.03,30.74,18.41
0,Consumer,1161401.34,134119.21,5191,1003,11.55,50.56,19.32


In [4]:
subcategory_results = profitability_summary(
    df, "Sub-Category"
)

display(subcategory_results.round(2))

,Sub-Category,Sales,Profit,Records,Loss_Making_Records,Profit_Margin_%,Sales_Share_%,Loss_Record_Share_%
16,Tables,206965.53,-17725.48,319,203,-8.56,9.01,63.64
4,Bookcases,114880.00,-3472.56,228,109,-3.02,5.00,47.81
15,Supplies,46673.54,-1189.10,190,33,-2.55,2.03,17.37
8,Fasteners,3024.28,949.52,217,12,31.40,0.13,5.53
11,Machines,189238.63,3384.76,115,44,1.79,8.24,38.26
10,Labels,12486.31,5546.25,364,0,44.42,0.54,0.00
2,Art,27118.79,6527.79,796,0,24.07,1.18,0.00
7,Envelopes,16476.40,6964.18,254,0,42.27,0.72,0.00
9,Furnishings,91705.16,13059.14,957,167,14.24,3.99,17.45
1,Appliances,107532.16,18138.01,466,67,16.87,4.68,14.38


Preliminary Profitability Findings

Furniture: Generated 32.3% of sales but only 6.4% of total profit, with a 2.49% profit margin

Product Sub-Categories: Tables recorded a $17,725 loss, while Bookcases recorded a $3,473 loss

Geographic Performance: The Central region had the lowest profit margin (7.92%) and the highest proportion of loss-making records (31.90%)

Customer Segments: Consumer customers contributed 50.6% of sales but had a lower profit margin (11.55%) than Corporate (13.03%) and Home Office (14.03%) customers